# Inference Callbacks

This notebook demonstrates the **callback system** for the inference pipeline.
We train a real ACT policy, export it to ONNX, then use callbacks to add
timing, throughput monitoring, safety clamping, and lifecycle tracking — all
without modifying model or runner code.

**Sections:**
1. Train & export an ACT policy
2. Build an observation from the dataset
3. Baseline inference (no callbacks)
4. `Timer` — measure prediction latency
5. `ThroughputMonitor` — track predictions per second
6. Custom callback — clamp actions to safe limits
7. Composing multiple callbacks
8. Full lifecycle hooks (`on_load`, `on_predict_start`, `on_predict_end`, `on_reset`)
9. Context manager (`with` statement)

**Requirements:** CPU is sufficient — `fast_dev_run=1` trains a single batch.

## 1. Train & Export

In [1]:
# Copyright (C) 2026 Intel Corporation
# SPDX-License-Identifier: Apache-2.0

import tempfile
from pathlib import Path

from physicalai.data import LeRobotDataModule
from physicalai.policies import get_policy
from physicalai.train import Trainer

policy = get_policy("act", source="physicalai")

datamodule = LeRobotDataModule(
    repo_id="lerobot/pusht",
    train_batch_size=8,
    episodes=list(range(10)),
)

trainer = Trainer(
    fast_dev_run=1,
    enable_checkpointing=False,
    logger=False,
    enable_progress_bar=False,
)

trainer.fit(policy, datamodule=datamodule)

export_dir = Path(tempfile.mkdtemp(prefix="act_callback_demo_"))
policy.export(export_dir, "onnx")

print("Exported files:")
for f in sorted(export_dir.iterdir()):
    print(f"  {f.name:30s} {f.stat().st_size:>12,} bytes")

/home/sakcay/projects/physical-ai/physical-ai-studio/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.
You are using a CUDA device ('NVIDIA RTX A6000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/gener

┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ val_rollout   │ Rollout         │      0 │ train │     0 │
│ 1 │ test_rollout  │ Rollout         │      0 │ train │     0 │
│ 2 │ _preprocessor │ ACTPreprocessor │      0 │ train │     0 │
│ 3 │ model         │ ACT             │ 51.6 M │ train │     0 │
└───┴───────────────┴─────────────────┴────────┴───────┴───────┘

Trainable params: 51.6 M                                                                                           
Non-trainable params: 18                                                                                           
Total params: 51.6 M                                                                                               
Total estimated model params size (MB): 206                                                                        
Modules in train mode: 191                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/sakcay/projects/physical-ai/physical-ai-studio/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/sakcay/projects/physical-ai/physical-ai-studio/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/sakcay/projects/physical-ai/physical-ai-studio/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:106: Total length of `DataLoader` across ranks is zero. Please make sure this was your intention.
/home/sakcay/projects/physical-ai/physical-ai-studio/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:378: You have overridden `transfer_batc

[torch.onnx] Obtain model graph for `ACT([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ACT([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Applied 26 of general pattern rewrite rules.
Exported files:
  act.onnx                            703,846 bytes
  act.onnx.data                   137,166,848 bytes
  manifest.json                           270 bytes
  metadata.yaml                         2,154 bytes


## 2. Build an Observation

In [2]:
batch = next(iter(datamodule.train_dataloader()))
observation = batch[0:1].to_numpy().to_dict(flatten=False)

print("Observation keys:")
for key, val in observation.items():
    if hasattr(val, "shape"):
        print(f"  {key}: shape={val.shape}, dtype={val.dtype}")
    elif isinstance(val, dict):
        for k2, v2 in val.items():
            if hasattr(v2, "shape"):
                print(f"  {key}.{k2}: shape={v2.shape}, dtype={v2.dtype}")

Observation keys:
  action: shape=(1, 100, 2), dtype=float32
  state: shape=(1, 2), dtype=float32
  images: shape=(1, 3, 96, 96), dtype=float32
  episode_index: shape=(1,), dtype=int64
  frame_index: shape=(1,), dtype=int64
  index: shape=(1,), dtype=int64
  task_index: shape=(1,), dtype=int64
  timestamp: shape=(1,), dtype=float32
  extra.next.reward: shape=(1,), dtype=float32
  extra.next.done: shape=(1,), dtype=bool
  extra.next.success: shape=(1,), dtype=bool
  extra.action_is_pad: shape=(1, 100), dtype=bool


## 3. Baseline — No Callbacks

In [3]:
from physicalai.inference import InferenceModel

model = InferenceModel.load(export_dir)
model.reset()
action = model.select_action(observation)

print(f"Model:        {model!r}")
print(f"Callbacks:    {model.callbacks}")
print(f"Action shape: {action.shape}")
print(f"Action dtype: {action.dtype}")
print(f"Action range: [{action.min():.2f}, {action.max():.2f}]")

assert model.callbacks == []
assert action.shape[-1] == 2

2026-03-25 07:31:55.133235353 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


Model:        InferenceModel(policy=act, backend=onnx, device=cpu, runner=SinglePass())
Callbacks:    []
Action shape: (1, 100, 2)
Action dtype: float32
Action range: [219.27, 275.95]


## 4. Timer — Measure Prediction Latency

In [4]:
from physicalai.inference.callbacks import Timer

timer = Timer()
model = InferenceModel.load(export_dir, callbacks=[timer])

for _ in range(5):
    model.select_action(observation)

print(f"Last prediction: {timer.last_duration_ms:.2f} ms")
print(f"All durations:   {[f'{d:.2f}' for d in timer.history]}")
print(f"Average latency: {sum(timer.history) / len(timer.history):.2f} ms")
print(f"Timer repr:      {timer!r}")

assert len(timer.history) == 5
assert all(d > 0 for d in timer.history)

Last prediction: 4.29 ms
All durations:   ['7.85', '7.15', '5.52', '5.16', '4.29']
Average latency: 5.99 ms
Timer repr:      Timer(last=4.3ms, calls=5)


## 5. ThroughputMonitor — Track Predictions per Second

In [ ]:
from physicalai.inference.callbacks import ThroughputMonitor

monitor = ThroughputMonitor(window_seconds=5.0)
model = InferenceModel.load(export_dir, callbacks=[monitor])

for _ in range(10):
    model.select_action(observation)

print(f"Throughput:          {monitor.throughput:.1f} predictions/sec")
print(f"Total calls:         {monitor.total_calls}")
print(f"Monitor repr:        {monitor!r}")

assert monitor.total_calls == 10
assert monitor.throughput > 0

## 6. Custom Callback — Clamp Actions to Safe Limits

Subclass `Callback` and override `on_predict_end` to modify outputs.
Returning a dict from a hook replaces the pipeline data.

In [6]:
from typing import Any, override

import numpy as np

from physicalai.inference import Callback


class SafetyClampCallback(Callback):
    """Clamp all actions to safe joint limits."""

    def __init__(self, low: float = -1.0, high: float = 1.0) -> None:
        self.low = low
        self.high = high
        self.clamp_count = 0

    @override
    def on_predict_end(self, outputs: dict[str, Any]) -> dict[str, Any]:
        action = outputs["action"]
        clamped = np.clip(action, self.low, self.high)
        if not np.array_equal(action, clamped):
            self.clamp_count += 1
        return {**outputs, "action": clamped}

    def __repr__(self) -> str:
        return f"SafetyClampCallback(low={self.low}, high={self.high}, clamped={self.clamp_count})"


# Baseline (unclamped)
raw_model = InferenceModel.load(export_dir)
raw_action = raw_model.select_action(observation)

# With safety clamp
safety = SafetyClampCallback(low=100.0, high=200.0)
clamped_model = InferenceModel.load(export_dir, callbacks=[safety])
clamped_action = clamped_model.select_action(observation)

print(f"Raw action range:     [{raw_action.min():.2f}, {raw_action.max():.2f}]")
print(f"Clamped action range: [{clamped_action.min():.2f}, {clamped_action.max():.2f}]")
print(f"Times clamped:        {safety.clamp_count}")
print(f"Safety repr:          {safety!r}")

assert clamped_action.min() >= 100.0
assert clamped_action.max() <= 200.0

Raw action range:     [219.27, 275.95]
Clamped action range: [200.00, 200.00]
Times clamped:        1
Safety repr:          SafetyClampCallback(low=100.0, high=200.0, clamped=1)


## 7. Composing Multiple Callbacks

Pass a list of callbacks — they fire in declared order at each lifecycle point.

In [ ]:
timer2 = Timer()
monitor2 = ThroughputMonitor()
safety2 = SafetyClampCallback(low=100.0, high=200.0)

model = InferenceModel.load(export_dir, callbacks=[timer2, monitor2, safety2])

print(f"Registered: {model.callbacks}")

for _ in range(5):
    model.select_action(observation)

print(f"Action range:   [{clamped_action.min():.2f}, {clamped_action.max():.2f}]")
print(f"Latency (last): {timer2.last_duration_ms:.2f} ms")
print(f"Throughput:     {monitor2.throughput:.1f} predictions/sec")

assert timer2.last_duration_ms > 0.0
assert monitor2.throughput > 0
assert safety2.clamp_count > 0

## 8. Full Lifecycle Hooks

All four hooks fire in order: `on_load` → `on_predict_start` → `on_predict_end` → `on_reset`.

In [7]:
class LifecycleCallback(Callback):
    """Record all lifecycle events in order."""

    def __init__(self) -> None:
        self.events: list[str] = []

    @override
    def on_load(self, model: InferenceModel) -> None:
        self.events.append(f"loaded:{model.policy_name}")

    @override
    def on_predict_start(self, inputs: dict[str, Any]) -> None:
        self.events.append("predict_start")

    @override
    def on_predict_end(self, outputs: dict[str, Any]) -> None:
        self.events.append("predict_end")

    @override
    def on_reset(self) -> None:
        self.events.append("reset")


lifecycle = LifecycleCallback()
model = InferenceModel.load(export_dir, callbacks=[lifecycle])
print(f"After load:    {lifecycle.events}")

model.select_action(observation)
print(f"After predict: {lifecycle.events}")

model.reset()
print(f"After reset:   {lifecycle.events}")

assert lifecycle.events == ["loaded:act", "predict_start", "predict_end", "reset"]

After load:    ['loaded:act']
After predict: ['loaded:act', 'predict_start', 'predict_end']
After reset:   ['loaded:act', 'predict_start', 'predict_end', 'reset']


## 9. Context Manager

`InferenceModel` supports `with` statements — `reset()` is called on entry and exit.

In [8]:
timer3 = Timer()

with InferenceModel.load(export_dir, callbacks=[timer3]) as model:
    model.reset()
    action1 = model.select_action(observation)
    action2 = model.select_action(observation)

print(f"Predictions inside `with`: {len(timer3.history)}")
print(f"Durations: {[f'{d:.2f} ms' for d in timer3.history]}")
print(f"Action shapes: {action1.shape}, {action2.shape}")

assert len(timer3.history) == 2
assert action1.shape[-1] == 2
assert action2.shape[-1] == 2

Predictions inside `with`: 2
Durations: ['8.10 ms', '6.90 ms']
Action shapes: (1, 100, 2), (1, 100, 2)
